# Lab 4

### Setup
Load the libraries and environment.

In [1]:
import os

os.environ['MUJOCO_PATH'] = '/usr'
os.environ['CMAKE_PREFIX_PATH'] = '/usr'
os.environ['LD_LIBRARY_PATH'] = '/usr/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

%pip install numpy==2.2.6 gymnasium mujoco tqdm torch torchvision 'stable-baselines3[extra]' moviepy ipywidgets --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gymnasium as gym
import moviepy
import numpy as np
import os
import torch
from pathlib import Path
from stable_baselines3 import SAC, PPO, __version__ as sb3_version
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.logger import configure


if torch.cuda.is_available():
    device = "cuda"
elif torch.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(
    f"Gymnasium: {gym.__version__}\n"
    f"SB3: {sb3_version}\n"
    f"Torch: {torch.__version__}\n"
    f"Device: {device}\n"
    f"MoviePy: {moviepy.__version__}"
)

# Environment configuration
ENV_NAME = "Reacher-v5"

# Training hyperparameters
TOTAL_TRAINING_STEPS = 1_000_000
LEARNING_RATE = 3e-4
GAMMA = 0.99
VERBOSE = 0

# SAC-specific hyperparameters
SAC_BUFFER_SIZE = 1_000_000
SAC_BATCH_SIZE = 256
SAC_TAU = 0.005

# PPO-specific hyperparameters
NUM_PPO_STEPS = 2048
PPO_BATCH_SIZE = 64
GAE_LAMBDA = 0.95
NUM_PPO_EPOCHS = 10
PPO_CLIP_RANGE = 0.2

# Evaluation configuration
RECORD_VIDEO = True
MAX_RENDER_STEPS = 2000  # Prevent infinite loops

def make_env():
    env = gym.make(ENV_NAME)
    env = Monitor(env)
    return env

Gymnasium: 1.2.2
SB3: 2.7.0
Torch: 2.9.0+cu128
Device: cuda
MoviePy: 2.1.2


In [3]:
vec_env_sac = DummyVecEnv([make_env])
vec_env_sac = VecNormalize(vec_env_sac, norm_obs=True, norm_reward=False)

if Path("sac_reacher_model.zip").exists():
    model_sac = SAC.load("sac_reacher_model", vec_env_sac)
else:
    logdir_sac = Path("runs") / "sac_reacher"
    logdir_sac.mkdir(parents=True, exist_ok=True)

    model_sac = SAC(
        "MlpPolicy",
        vec_env_sac,
        verbose=VERBOSE,
        tensorboard_log=str(logdir_sac),  # tensorboard --logdir ./runs to visualize
        learning_rate=LEARNING_RATE,
        buffer_size=SAC_BUFFER_SIZE,
        batch_size=SAC_BATCH_SIZE,
        tau=SAC_TAU,
        gamma=GAMMA,
        target_entropy="auto",
        device=device,
    )

    new_logger = configure(str(logdir_sac), ["tensorboard"])
    model_sac.set_logger(new_logger)

    model_sac.learn(total_timesteps=TOTAL_TRAINING_STEPS, progress_bar=True)
    model_sac.save("sac_reacher_model")
    
    vec_env_sac.save("vecnormalize_reacher.pkl")
    vec_env_sac.close()

In [4]:
vec_env_ppo = DummyVecEnv([make_env])
vec_env_ppo = VecNormalize(vec_env_ppo, norm_obs=True, norm_reward=False)

if Path("sac_reacher_model.zip").exists():
    model_sac = SAC.load("sac_reacher_model", vec_env_sac)
else:
    logdir_sac = Path("runs") / "sac_reacher"
    logdir_sac.mkdir(parents=True, exist_ok=True)

    model_sac = SAC(
        "MlpPolicy",
        vec_env_sac,
        verbose=VERBOSE,
        tensorboard_log=str(logdir_sac),  # tensorboard --logdir ./runs to visualize
        learning_rate=LEARNING_RATE,
        buffer_size=SAC_BUFFER_SIZE,
        batch_size=SAC_BATCH_SIZE,
        tau=SAC_TAU,
        gamma=GAMMA,
        target_entropy="auto",
        device=device,
    )

    new_logger = configure(str(logdir_sac), ["tensorboard"])
    model_sac.set_logger(new_logger)

    model_sac.learn(total_timesteps=TOTAL_TRAINING_STEPS, progress_bar=True)
    model_sac.save("sac_reacher_model")
    
    vec_env_sac.save("vecnormalize_reacher.pkl")
    vec_env_sac.close()

In [7]:
import numpy as np
import gymnasium as gym
from stable_baselines3 import SAC, PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import os
import imageio


def evaluate_model(model_path, vec_path, n_episodes=10, record_video=RECORD_VIDEO):
    algo_name = "sac" if "sac" in model_path.lower() else "ppo"
    
    # Create and wrap environment
    env = DummyVecEnv([lambda: gym.make(ENV_NAME, render_mode="rgb_array")])
    env = VecNormalize.load(vec_path, env)
    env.training = False
    env.norm_reward = False
    
    # Load model
    model = SAC.load(model_path, env=env) if algo_name == "sac" else PPO.load(model_path, env=env)
    
    episode_rewards = []
    frames = []
    
    for episode in range(1, n_episodes + 1):
        obs = env.reset()
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < MAX_RENDER_STEPS:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated = env.step(action)
            
            # Capture frame for first episode only
            if record_video:
                frame = env.envs[0].render()
                if frame is not None:
                    frames.append(frame)
            
            total_reward += float(np.sum(reward))
            
            done = np.any(terminated)
            steps += 1
        
        episode_rewards.append(total_reward)
        print(f"Episode {episode} finished after {steps} steps, reward: {total_reward:.2f}")
    
    # Save video if frames were collected
    if record_video and len(frames) > 0:
        video_folder = f"videos/{algo_name}_eval"
        os.makedirs(video_folder, exist_ok=True)
        video_path = f"{video_folder}/{algo_name}_reacher_eval.mp4"
        
        print(f"Saving video with {len(frames)} frames...")
        imageio.mimsave(video_path, frames, fps=30)
        print(f"✅ {algo_name.upper()} video saved to: {video_path}")
    elif record_video:
        print(f"⚠️ No frames captured for {algo_name.upper()}")
    
    env.close()
    return np.mean(episode_rewards)


# Run evaluation with video recording
print("🎬 Recording SAC video...")
sac_reward = evaluate_model("sac_reacher_model.zip", "vecnormalize_reacher.pkl", record_video=True)

print("\n🎬 Recording PPO video...")
ppo_reward = evaluate_model("ppo_reacher_model.zip", "vecnormalize_reacher_ppo.pkl", record_video=True)

print(f"\n{'='*50}")
print(f"SAC mean reward: {sac_reward:.2f}")
print(f"PPO mean reward: {ppo_reward:.2f}")
print(f"{'='*50}")

🎬 Recording SAC video...



(python:71822): Gtk-WARNING **: 11:43:31.341: gtk_disable_setlocale() must be called before gtk_init()


Episode 1 finished after 50 steps, reward: -3.28
Episode 2 finished after 50 steps, reward: -2.30
Episode 3 finished after 50 steps, reward: -4.15
Episode 4 finished after 50 steps, reward: -3.42
Episode 5 finished after 50 steps, reward: -3.44
Episode 6 finished after 50 steps, reward: -4.29
Episode 7 finished after 50 steps, reward: -1.62
Episode 8 finished after 50 steps, reward: -3.02
Episode 9 finished after 50 steps, reward: -3.74
Episode 10 finished after 50 steps, reward: -3.75
Saving video with 500 frames...
✅ SAC video saved to: videos/sac_eval/sac_reacher_eval.mp4

🎬 Recording PPO video...



(python:71822): Gtk-WARNING **: 11:43:36.324: gtk_disable_setlocale() must be called before gtk_init()


Episode 1 finished after 50 steps, reward: -3.06
Episode 2 finished after 50 steps, reward: -4.22
Episode 3 finished after 50 steps, reward: -3.91
Episode 4 finished after 50 steps, reward: -4.44
Episode 5 finished after 50 steps, reward: -4.37
Episode 6 finished after 50 steps, reward: -3.59
Episode 7 finished after 50 steps, reward: -4.42
Episode 8 finished after 50 steps, reward: -2.41
Episode 9 finished after 50 steps, reward: -1.90
Episode 10 finished after 50 steps, reward: -7.00
Saving video with 500 frames...
✅ PPO video saved to: videos/ppo_eval/ppo_reacher_eval.mp4

SAC mean reward: -3.30
PPO mean reward: -3.93
